# Thai Text Classification - Neural Approach
Uses PyThaiNLP for tokenization and TensorFlow/Keras (Word Embeddings, LSTM) for Deep Learning classification.

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# PyThaiNLP for Thai language tokenization
from pythainlp.tokenize import word_tokenize



In [10]:
# --- METHOD B: Neural Approach (Word Embedding + LSTM) ---
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout


# Load and Preprocess Data


In [15]:
df = pd.read_csv('labeled_songs.csv')
print("Dataset loaded successfully.")

# Extract texts and labels
texts = df['lyric'].astype(str).tolist()
labels = df['label'].tolist()

# Encode labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(labels)
num_classes = len(np.unique(y_encoded))

# Define a custom tokenizer using PyThaiNLP
def thai_tokenizer(text):
    return word_tokenize(text, engine='newmm')

Dataset loaded successfully.


# METHOD B: Neural Approach

In [ ]:
print("\n--- Method B: Neural Approach (Word Embedding + LSTM) ---")

# Tokenize using PyThaiNLP first for all texts
tokenized_texts = [' '.join(thai_tokenizer(text)) for text in texts]

# Feature Extraction: Keras Tokenizer to create integer sequences
max_words = 5000  # Max number of words in vocabulary
max_len = 50      # Max length of each sequence

keras_tokenizer = Tokenizer(num_words=max_words)
keras_tokenizer.fit_on_texts(tokenized_texts)
sequences = keras_tokenizer.texts_to_sequences(tokenized_texts)

# Pad sequences
X_padded = pad_sequences(sequences, maxlen=max_len)

# Train-test split
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X_padded, y_encoded, test_size=0.2, random_state=42)

# One-hot encode labels for the neural network if it's multiclass, or leave as is for binary
# Assuming binary or multiclass classification
y_train_nn = tf.keras.utils.to_categorical(y_train_n, num_classes)
y_test_nn = tf.keras.utils.to_categorical(y_test_n, num_classes)

# Model: LSTM Deep Learning Model

In [ ]:
embedding_dim = 64

neural_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax' if num_classes > 2 else 'sigmoid')
])

# Compile model
loss_function = 'categorical_crossentropy' if num_classes > 2 else 'binary_crossentropy'
# Note: if binary, we only need to output 1 unit instead of num_classes, but keeping it general for categorical
if num_classes == 2:
    neural_model = Sequential([
        Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
        LSTM(64),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    loss_function = 'binary_crossentropy'
    y_train_nn = y_train_n
    y_test_nn = y_test_n

neural_model.compile(optimizer='adam', loss=loss_function, metrics=['accuracy'])

# Train Model
print("Training Neural Model...")
# Small epochs/batch_size due to potentially small local datasets
history = neural_model.fit(X_train_n, y_train_nn, epochs=5, batch_size=32, validation_data=(X_test_n, y_test_nn), verbose=1)

# Evaluation
loss, accuracy = neural_model.evaluate(X_test_n, y_test_nn, verbose=0)
print(f"Neural Model Accuracy: {accuracy:.4f}")